# Script 1 — Data Processing & Master Catalogue Generation

This notebook merges and curates three input sources — the Exoplanet.eu catalogue,
the NASA Exoplanet Archive, and the CIF25 stellar catalogue — into a single clean
master catalogue with unified distances, propagated uncertainties, manual literature
corrections, and full reference traceability per parameter.

**Author:** Rafael Carmona Vendoiro, UCM  
**Last updated:** 22/04/2026  
**Output:** `Exoplanets_SolarNeighbourhood_Catalogue.csv`

In [12]:
from pathlib import Path

import pandas as pd
import numpy as np
from astropy.io import fits
from astropy import units as u, constants as const

## Inputs and Outputs

All file paths used by the notebook are defined here so the pipeline can be updated from a single place.
If any source file is renamed or moved, only this cell should need to be edited.

In [13]:
# Input catalogues
file_csv_raw   = Path('eu_actualizado_cif2025.csv')
file_new       = Path('nuevos_planetas.csv')
file_fits_cif  = Path('eu_cif2025corr.fit')
file_gaia      = Path('distancias_gaia.csv')
file_discovery = Path('discovery_refs.csv')
file_patches   = Path('correcciones_manuales.csv')

# Output catalogue
file_output    = Path('Exoplanets_SolarNeighbourhood_Catalogue.csv')

## Step 1 — Load and Merge Catalogues (EU + NASA / New Planets)

In [14]:
print("1. Loading and merging catalogues...")
df = pd.read_csv(file_csv_raw)
df['_is_new'] = False

try:
    # Try comma separator first; fall back to semicolon if needed (common in Excel exports)
    try:
        df_new = pd.read_csv(file_new, sep=',')
        if len(df_new.columns) < 2:
            raise ValueError("Too few columns — trying semicolon separator.")
    except (ValueError, Exception):
        df_new = pd.read_csv(file_new, sep=';')

    df_new['_is_new'] = True
    df = pd.concat([df, df_new], ignore_index=True)
    print(f"   -> {len(df_new)} extra planets added from '{file_new}'.")
except FileNotFoundError:
    print(f"   [!] '{file_new}' not found. Continuing with base catalogue only.")
except Exception as e:
    print(f"   [!] Error reading '{file_new}': {e}")

cif_names_set = set()
try:
    print("   -> Cross-matching with CIF25...")
    with fits.open(file_fits_cif) as hdul:
        data = hdul[1].data
        arr = np.array(data).byteswap().view(np.array(data).dtype.newbyteorder('='))
        df_cif = pd.DataFrame(arr)

    for col in df_cif.columns:
        if df_cif[col].dtype == object:
            try:
                df_cif[col] = df_cif[col].str.decode('utf-8').str.strip()
            except UnicodeDecodeError:
                pass

    mapping_cols = {'star_teff': 'Teff', 'star_mass': 'Mass_1',
                    'star_radius': 'Rad', 'star_sp_type': 'SpType'}
    mapping_errs = {'star_teff_error': 'e_Teff',
                    'star_mass_error': 'e_Mass',
                    'star_radius_error': 'e_Rad'}

    if 'name_2' in df_cif.columns:
        df_cif_indexed = df_cif.set_index('name_2')
        for idx_csv, row_csv in df.iterrows():
            name_p = str(row_csv['name']).strip()
            if name_p in df_cif_indexed.index:
                match = df_cif_indexed.loc[name_p]
                row_fits = match.iloc[0] if isinstance(match, pd.DataFrame) else match
                cif_names_set.add(name_p)

                for col_csv, col_fits in mapping_cols.items():
                    if col_fits in row_fits and pd.notna(row_fits[col_fits]):
                        df.at[idx_csv, col_csv] = row_fits[col_fits]

                for col_base, col_err in mapping_errs.items():
                    if col_err in row_fits and pd.notna(row_fits[col_err]):
                        df.at[idx_csv, f'{col_base}_min'] = row_fits[col_err]
                        df.at[idx_csv, f'{col_base}_max'] = row_fits[col_err]

        print(f"   -> {len(cif_names_set)} systems cross-matched with CIF25.")
except Exception as e:
    print(f"   ERROR reading FITS file: {e}")

1. Loading and merging catalogues...
   -> 11 extra planets added from 'nuevos_planetas.csv'.
   -> Cross-matching with CIF25...
   -> 78 systems cross-matched with CIF25.


## Step 2 — Reference Traceability Layer (NASA vs EU vs CIF25)

For each tracked parameter, every row is tagged with a source hyperlink
(colour-coded: blue = CIF25, pink = Exoplanet.eu, orange = NASA).

In [15]:
print("2. Building base reference traceability layer...")

tracked_parameters = [
    'orbital_period', 'radius', 'mass', 'semi_major_axis', 'eccentricity',
    'inclination', 'tzero_tr', 'tperi', 'omega', 'star_radius',
    'star_mass', 'star_teff', 'star_distance', 'star_sp_type'
]

LINK_CIF = (
    '<a href="https://ui.adsabs.harvard.edu/abs/2025A%26A...693A.228C/abstract" '
    'target="_blank" style="text-decoration:none; color:#007bff; font-weight:bold;">Cif25</a>'
)

for param in tracked_parameters:
    col_ref = f"{param}_ref"
    if col_ref not in df.columns:
        df[col_ref] = ""

    for idx, row in df.iterrows():
        p_name     = str(row['name']).strip()
        is_stellar = param.startswith('star_')

        # Planets sourced from the NASA Exoplanet Archive
        if row.get('_is_new') == True:
            ref_text = str(row.get('referencia_general', 'NASA')).strip()
            if ref_text in ('nan', ''):
                ref_text = 'NASA'

            if '<a href' not in ref_text:
                ref_final = (
                    f'<a href="https://exoplanetarchive.ipac.caltech.edu/" '
                    f'target="_blank" style="text-decoration:none; color:#ff7b00; '
                    f'font-weight:bold;">{ref_text}</a>'
                )
            else:
                ref_final = ref_text
            df.at[idx, col_ref] = ref_final

        # Planets from Exoplanet.eu (stellar params overridden by CIF25 where available)
        else:
            if is_stellar and p_name in cif_names_set:
                df.at[idx, col_ref] = LINK_CIF
            else:
                slug = p_name.lower().replace(' ', '_').replace('+', '_').replace('-', '_')
                df.at[idx, col_ref] = (
                    f'<a href="http://exoplanet.eu/catalog/{slug}/" '
                    f'target="_blank" style="text-decoration:none; color:#d63384; '
                    f'font-weight:bold;">Eu</a>'
                )

2. Building base reference traceability layer...


## Step 3 — Kinematic Distance Computation from SIMBAD (Gaia / Hipparcos Parallaxes)

In [16]:
print("3. Injecting parallaxes and recomputing distances...")
try:
    df_gaia  = pd.read_csv(file_gaia)
    gaia_dict = df_gaia.set_index('nombre_estrella').to_dict('index')
    updated_systems = 0

    for idx, row in df.iterrows():
        p_name = str(row['name']).strip()
        s_name = str(row.get('star_name', '')).strip()
        if not s_name or s_name == 'nan':
            s_name = p_name.rsplit(' ', 1)[0]

        if s_name in gaia_dict:
            plx      = float(gaia_dict[s_name]['parallax'])
            e_plx    = float(gaia_dict[s_name]['parallax_error'])
            ref_html = str(gaia_dict[s_name].get('parallax_ref', ''))

            if plx > 0:
                d   = 1000.0 / plx        # distance in pc
                e_d = d * (e_plx / plx)   # propagated 1-sigma uncertainty

                df.at[idx, 'star_distance']           = d
                df.at[idx, 'star_distance_error_min'] = e_d
                df.at[idx, 'star_distance_error_max'] = e_d

                if ref_html and ref_html != 'nan':
                    df.at[idx, 'star_distance_ref'] = ref_html
                updated_systems += 1

    print(f"   -> Distances computed for {updated_systems} stellar systems.")
except FileNotFoundError:
    print("   [!] 'distancias_gaia.csv' not found. Original catalogue distances will be kept.")

3. Injecting parallaxes and recomputing distances...
   -> Distances computed for 132 stellar systems.


### Step 3.2 — Discovery Reference Injection

In [17]:
print("3.2. Injecting discovery references...")
try:
    try:
        df_disc = pd.read_csv(file_discovery, sep=';', skipinitialspace=True, encoding='utf-8-sig')
        if 'name' not in [c.strip().lower() for c in df_disc.columns]:
            raise ValueError("Column 'name' not found with semicolon separator.")
    except (ValueError, Exception):
        df_disc = pd.read_csv(file_discovery, sep=',', skipinitialspace=True, encoding='utf-8-sig')

    df_disc.columns = [c.strip().lower() for c in df_disc.columns]
    disc_dict = df_disc.set_index('name')['discovery_ref'].to_dict()

    df['discovery_ref'] = df['name'].map(disc_dict).fillna('')
    found = df['discovery_ref'].astype(bool).sum()
    print(f"   -> Discovery references injected: {found} of {len(df)} planets.")
except FileNotFoundError:
    df['discovery_ref'] = ''
    print(f"   [!] '{file_discovery}' not found. Column 'discovery_ref' initialised as empty.")
except Exception as e:
    df['discovery_ref'] = ''
    print(f"   [!] Error processing discovery references: {e}")

3.2. Injecting discovery references...
   -> Discovery references injected: 130 of 132 planets.


## Step 4 — Manual Literature Corrections (Patch Catalogue)

Corrections are loaded from `correcciones_manuales.csv`. Each row specifies a planet name
(or a `|`-separated batch), an action (`borrar` / `modificar`), and optionally new values
and references. The helper function `apply_or_clear` is defined once here and reused per patch.

In [18]:
def apply_or_clear(dataframe, column, value, index):
    """
    Write `value` into dataframe[column] at `index`.
    Sets the cell to NaN if value is 'DELETE'; skips if value is empty or NaN.
    """
    if pd.isna(value) or str(value).strip() == '':
        return
    if column not in dataframe.columns:
        dataframe[column] = np.nan

    if str(value).strip().upper() in ('BORRAR', 'DELETE'):
        dataframe.at[index, column] = np.nan
    else:
        try:
            dataframe.at[index, column] = float(value)
        except ValueError:
            if pd.api.types.is_numeric_dtype(dataframe[column]):
                dataframe[column] = dataframe[column].astype('object')
            dataframe.at[index, column] = str(value)


print("4. Applying manual corrections from the literature...")
try:
    try:
        df_patches = pd.read_csv(file_patches, sep=';', skipinitialspace=True, encoding='utf-8-sig')
        if 'name' not in [c.strip().lower() for c in df_patches.columns]:
            raise ValueError("Column 'name' not found with semicolon separator.")
    except (ValueError, Exception):
        df_patches = pd.read_csv(file_patches, sep=',', skipinitialspace=True, encoding='utf-8-sig')

    df_patches.columns = [c.strip().lower() for c in df_patches.columns]

    deleted, modified = 0, 0

    for _, patch_row in df_patches.iterrows():
        if pd.isna(patch_row.get('name')) or pd.isna(patch_row.get('accion')):
            continue

        # Multiple planets can be batched in a single row, separated by '|'
        planet_names = [n.strip() for n in str(patch_row['name']).split('|')]
        action       = str(patch_row['accion']).strip().lower()

        if action == 'borrar':
            for p_name in planet_names:
                df = df[df['name'] != p_name]
                deleted += 1
                print(f"   [-] Removed: {p_name}")
            continue

        if action == 'modificar':
            param = str(patch_row.get('parametro', '')).strip()
            if not param:
                continue

            new_val     = patch_row.get('nuevo_valor',     np.nan)
            new_err_min = patch_row.get('nuevo_error_min', np.nan)
            new_err_max = patch_row.get('nuevo_error_max', np.nan)
            new_ref     = patch_row.get('nueva_ref',       np.nan)

            for p_name in planet_names:
                idx_list = df[df['name'] == p_name].index
                if len(idx_list) == 0:
                    continue
                idx_df = idx_list[0]

                apply_or_clear(df, param,                new_val,     idx_df)
                apply_or_clear(df, f"{param}_error_min", new_err_min, idx_df)
                apply_or_clear(df, f"{param}_error_max", new_err_max, idx_df)
                apply_or_clear(df, f"{param}_ref",       new_ref,     idx_df)

                modified += 1

    df.reset_index(drop=True, inplace=True)
    print(f"   -> Summary: {deleted} removed | {modified} parameters patched.")
except Exception as e:
    print(f"   [!] Error processing patches: {e}")

4. Applying manual corrections from the literature...
   [-] Removed: Wolf 359 c
   [-] Removed: HD 219134 e
   -> Summary: 2 removed | 1472 parameters patched.


## Step 5 — Derived Parameter Recalculation & Export

In [19]:
print("5. Recomputing derived parameters and exporting...")

# Standardise planet status field
if 'planet_status' not in df.columns:
    df['planet_status'] = 'Confirmed'
df['planet_status'] = df['planet_status'].fillna('Confirmed').str.title()

df['detection_type']          = df['detection_type'].fillna('Unknown')
df['mass_measurement_type']   = df['mass_measurement_type'].fillna('Unknown')
df['radius_measurement_type'] = df['radius_measurement_type'].fillna('Unknown')
df['star_sp_type']            = df['star_sp_type'].fillna('')

def classify_method(row):
    det  = str(row['detection_type']).lower()
    mass = str(row['mass_measurement_type']).lower()

    has_rv      = 'radial velocity' in det  or 'rv' in det  or 'radial velocity' in mass
    has_transit = 'transit' in det and 'timing' not in det and 'ttv' not in det
    has_astro   = 'astrometry' in det or 'astrometry' in mass
    has_ttv     = 'ttv' in det or 'timing' in det or 'ttv' in mass or 'timing' in mass

    if has_ttv:                                          return 'TTV'
    elif has_transit and has_rv:                         return 'Transit + RV'
    elif has_rv and has_astro:                           return 'RV + Astrometry'
    elif has_rv and not has_transit and not has_astro:   return 'RV'
    elif has_astro and not has_rv and not has_transit:   return 'Astrometry'
    elif has_transit and not has_rv:                     return 'Transit'
    else:                                                return 'Other'

df['Detection_Method_Full'] = df.apply(classify_method, axis=1)

M, R = df['star_mass'], df['star_radius']

# Stellar density in g/cm³
# ρ_☉ = M_☉ / (4/3 π R_☉³) ≈ 1.408 g/cm³
RHO_SUN = (const.M_sun / (4/3 * np.pi * const.R_sun**3)).to(u.g / u.cm**3).value

if 'star_density' not in df.columns:
    df['star_density'] = np.nan

density_calc     = (M / R**3) * RHO_SUN
df['star_density'] = df['star_density'].where(df['star_density'].notna(), density_calc)

# Uncertainty propagation in quadrature: σ_ρ/ρ = sqrt( (σ_M/M)² + (3·σ_R/R)² )
rel_M_up = df['star_mass_error_max'].abs().fillna(0) / M
rel_M_lo = df['star_mass_error_min'].abs().fillna(0) / M
rel_R_up = df['star_radius_error_max'].abs().fillna(0) / R
rel_R_lo = df['star_radius_error_min'].abs().fillna(0) / R

err_max_calc = df['star_density'] * np.sqrt(rel_M_up**2 + (3 * rel_R_lo)**2)
err_min_calc = df['star_density'] * np.sqrt(rel_M_lo**2 + (3 * rel_R_up)**2)

if 'star_density_error_max' not in df.columns:
    df['star_density_error_max'] = np.nan
if 'star_density_error_min' not in df.columns:
    df['star_density_error_min'] = np.nan

df['star_density_error_max'] = df['star_density_error_max'].where(
    df['star_density_error_max'].notna(), err_max_calc)
df['star_density_error_min'] = df['star_density_error_min'].where(
    df['star_density_error_min'].notna(), err_min_calc)

if 'star_density_ref' not in df.columns:
    df['star_density_ref'] = np.nan
df['star_density_ref'] = df['star_density_ref'].where(df['star_density_ref'].notna(), '-')

# Planetary quantities in Earth units (assuming input mass/radius are in Jupiter units)
MASS_JUP_TO_EARTH   = (const.M_jup / const.M_earth).value
RADIUS_JUP_TO_EARTH = (const.R_jup / const.R_earth).value

df['mass_earth']      = pd.to_numeric(df.get('mass', np.nan), errors='coerce') * MASS_JUP_TO_EARTH
df['mass_sini_earth'] = pd.to_numeric(df.get('mass_sini', np.nan), errors='coerce') * MASS_JUP_TO_EARTH
df['radius_earth']    = pd.to_numeric(df.get('radius', np.nan), errors='coerce') * RADIUS_JUP_TO_EARTH

# Propagate existing uncertainty columns to Earth units when available
for base_col, factor in (('mass', MASS_JUP_TO_EARTH), ('mass_sini', MASS_JUP_TO_EARTH), ('radius', RADIUS_JUP_TO_EARTH)):
    for err_suffix in ('_error_min', '_error_max'):
        src_col = f'{base_col}{err_suffix}'
        dst_col = f'{base_col}_earth{err_suffix}'
        if src_col in df.columns:
            df[dst_col] = pd.to_numeric(df[src_col], errors='coerce') * factor
        else:
            df[dst_col] = np.nan

# Reference epoch T0: transit midpoint, falling back to periastron time if unavailable
df['T0_val']     = df['tzero_tr'].fillna(df['tperi'])
df['T0_min']     = df['tzero_tr_error_min'].fillna(df['tperi_error_min'])
df['T0_max']     = df['tzero_tr_error_max'].fillna(df['tperi_error_max'])
df['T0_val_ref'] = df['tzero_tr_ref'].where(df['tzero_tr'].notna(), df['tperi_ref'])

# Equilibrium temperature following Perryman (2018).
# T_eq = T_eff * sqrt(R_star / (2a)) with R_star and a converted to the same units.
R_SUN_TO_AU = (const.R_sun / const.au).decompose().value
teff = pd.to_numeric(df['star_teff'], errors='coerce')
r_star = pd.to_numeric(df['star_radius'], errors='coerce')
semi_major_axis = pd.to_numeric(df['semi_major_axis'], errors='coerce')

def _numeric_or_nan(column_name):
    if column_name in df.columns:
        return pd.to_numeric(df[column_name], errors='coerce')
    return pd.Series(np.nan, index=df.index)

valid_teq = teff.notna() & r_star.notna() & semi_major_axis.notna() & (teff > 0) & (r_star > 0) & (semi_major_axis > 0)

if 'Teq_val' not in df.columns:
    df['Teq_val'] = np.nan
if 'Teq_error_min' not in df.columns:
    df['Teq_error_min'] = np.nan
if 'Teq_error_max' not in df.columns:
    df['Teq_error_max'] = np.nan

teq_calc = teff * np.sqrt((r_star * R_SUN_TO_AU) / (2.0 * semi_major_axis))

teff_err_min = _numeric_or_nan('star_teff_error_min').fillna(0)
teff_err_max = _numeric_or_nan('star_teff_error_max').fillna(0)
rstar_err_min = _numeric_or_nan('star_radius_error_min').fillna(0)
rstar_err_max = _numeric_or_nan('star_radius_error_max').fillna(0)
a_err_min = _numeric_or_nan('semi_major_axis_error_min').fillna(0)
a_err_max = _numeric_or_nan('semi_major_axis_error_max').fillna(0)

with np.errstate(divide='ignore', invalid='ignore'):
    teq_err_max_calc = teq_calc * np.sqrt(
        (teff_err_max / teff)**2 + 0.25 * (rstar_err_max / r_star)**2 + 0.25 * (a_err_min / semi_major_axis)**2
    )
    teq_err_min_calc = teq_calc * np.sqrt(
        (teff_err_min / teff)**2 + 0.25 * (rstar_err_min / r_star)**2 + 0.25 * (a_err_max / semi_major_axis)**2
    )

df['Teq_val'] = df['Teq_val'].where(df['Teq_val'].notna(), teq_calc)
df.loc[~valid_teq, ['Teq_val', 'Teq_error_min', 'Teq_error_max']] = np.nan
df.loc[valid_teq, 'Teq_error_min'] = teq_err_min_calc[valid_teq]
df.loc[valid_teq, 'Teq_error_max'] = teq_err_max_calc[valid_teq]

# Drop the internal merge helper column when it is still present.
if '_is_new' in df.columns:
    df = df.drop(columns=['_is_new'])

# ---------------------------------------------------------------------------
# Step 5b — Clean-reference and error cleanup
# If a parameter has no value (NaN or empty string), its _ref, _error_min
# and _error_max columns must also be blank. This prevents EU / NASA links
# and stale uncertainties from appearing next to absent values.
# ---------------------------------------------------------------------------

# Numeric parameters tracked with errors and a reference
numeric_params = [
    'mass',
    'mass_sini',
    'radius',
    'orbital_period',
    'semi_major_axis',
    'tzero_tr',
    'tperi',
    'star_radius',
    'star_mass',
    'star_distance',
    'star_density',
    'T0_val',
    'Teq_val',
]

# String/mixed parameters (NaN or empty string counts as absent)
string_params = [
    'eccentricity',
    'inclination',
    'omega',
    'star_teff',
    'star_sp_type',
]

for param in numeric_params:
    if param not in df.columns:
        continue
    mask = df[param].isna()
    for suffix in ('_ref', '_error_min', '_error_max'):
        col = f"{param}{suffix}"
        if col in df.columns:
            df.loc[mask, col] = np.nan if suffix != '_ref' else ''

for param in string_params:
    if param not in df.columns:
        continue
    mask = df[param].isna() | (df[param].astype(str).str.strip().isin(['', 'nan']))
    for suffix in ('_ref', '_error_min', '_error_max'):
        col = f"{param}{suffix}"
        if col in df.columns:
            df.loc[mask, col] = np.nan if suffix != '_ref' else ''

print("   -> Cleaned references and errors cleared.")

# Final export is deferred to Step 6, after the HZ columns are added.

5. Recomputing derived parameters and exporting...
   -> Cleaned references and errors cleared.


## Step 6 — Habitable Zone Classification (Kopparapu et al. 2014)

This section is dedicated only to habitable-zone physics.
It computes stellar luminosity from stellar parameters, evaluates the Kopparapu et al. (2014) HZ limits, and assigns each planet to a habitable-zone class.

Output columns from this step are the luminosity term (`hz_lum_solar`), the six HZ orbital boundaries, convenience aliases for conservative/optimistic limits, and `hz_zone`.

In [20]:
# ============================================================
# Habitable Zone classification (Kopparapu et al. 2014)
# Inputs: star_teff, star_radius, semi_major_axis
# This step only computes HZ-related quantities.
# ============================================================

print("6. Computing Habitable Zone limits (Kopparapu et al. 2014)...")

# ------------------------------------------------------------------
# Kopparapu (2014) polynomial coefficients for the six canonical limits.
# Index order: 0 = Recent Venus, 1 = Runaway Greenhouse, 2 = Maximum Greenhouse,
#              3 = Early Mars, 4 = Runaway GH for 5 M_earth, 5 = Runaway GH for 0.1 M_earth.
# ------------------------------------------------------------------
_HZ_SEFFSUN = np.array([1.776,      1.107,      0.356,      0.320,      1.188,      0.99  ])
_HZ_A       = np.array([2.136e-4,   1.332e-4,   6.171e-5,   5.547e-5,   1.433e-4,   1.209e-4])
_HZ_B       = np.array([2.533e-8,   1.580e-8,   1.698e-9,   1.526e-9,   1.707e-8,   1.404e-8])
_HZ_C       = np.array([-1.332e-11, -8.308e-12, -3.198e-12, -2.874e-12, -8.968e-12, -7.418e-12])
_HZ_D       = np.array([-3.097e-15, -1.931e-15, -5.575e-16, -5.011e-16, -2.084e-15, -1.713e-15])
_T_SUN      = 5778.0  # K


def _hz_seff_matrix(teff_array):
    """Return the S_eff matrix (N x 6) for an array of stellar effective temperatures."""
    t = (np.asarray(teff_array, dtype=float) - 5780.0)[:, np.newaxis]
    return _HZ_SEFFSUN + _HZ_A * t + _HZ_B * t**2 + _HZ_C * t**3 + _HZ_D * t**4


# 1) Stellar luminosity from the Stefan-Boltzmann law.
#    L/Lsun = (R/Rsun)^2 * (T_eff / T_sun)^4
teff = pd.to_numeric(df['star_teff'], errors='coerce')
r_star = pd.to_numeric(df['star_radius'], errors='coerce')
df['hz_lum_solar'] = r_star**2 * (teff / _T_SUN)**4

# 2) Evaluate HZ flux limits only within the calibrated temperature range.
in_range = teff.notna() & r_star.notna() & teff.between(2600.0, 7200.0)

n_oor = (~teff.between(2600.0, 7200.0) & teff.notna()).sum()
if n_oor:
    print(f"   [!] {n_oor} planets have T_eff outside 2600-7200 K. HZ columns will be NaN for those rows.")

seff_matrix = np.full((len(df), 6), np.nan)
seff_matrix[in_range.values] = _hz_seff_matrix(teff[in_range].values)

# 3) Convert flux limits to orbital distances in AU: d = sqrt(L/Lsun / S_eff).
lum_col = df['hz_lum_solar'].values[:, np.newaxis]
with np.errstate(invalid='ignore', divide='ignore'):
    hz_au = np.where(
        (lum_col > 0) & np.isfinite(lum_col) & np.isfinite(seff_matrix) & (seff_matrix > 0),
        np.sqrt(lum_col / seff_matrix),
        np.nan
    )

_HZ_COL_NAMES = [
    'hz_recent_venus_au',
    'hz_runaway_gh_au',
    'hz_max_gh_au',
    'hz_early_mars_au',
    'hz_rg_5me_au',
    'hz_rg_01me_au',
]
for i, col in enumerate(_HZ_COL_NAMES):
    df[col] = hz_au[:, i]

# 4) Convenience aliases used by downstream notebooks and exports.
df['hz_inner_conservative_au'] = df['hz_runaway_gh_au']
df['hz_outer_conservative_au'] = df['hz_max_gh_au']
df['hz_inner_optimistic_au'] = df['hz_recent_venus_au']
df['hz_outer_optimistic_au'] = df['hz_early_mars_au']

# 5) Classify each planet relative to the HZ boundaries using semi_major_axis.
a = pd.to_numeric(df['semi_major_axis'], errors='coerce')
rv = df['hz_recent_venus_au']
rg = df['hz_runaway_gh_au']
mg = df['hz_max_gh_au']
em = df['hz_early_mars_au']

no_data = a.isna() | rv.isna()
too_hot = ~no_data & (a < rv)
opt_inner = ~no_data & (a >= rv) & (a < rg)
cons_hz = ~no_data & (a >= rg) & (a <= mg)
opt_outer = ~no_data & (a > mg) & (a <= em)
too_cold = ~no_data & (a > em)

df['hz_zone'] = np.select(
    [no_data, too_hot, opt_inner, cons_hz, opt_outer, too_cold],
    ['No data', 'Too hot', 'Optimistic HZ (inner)', 'Conservative HZ', 'Optimistic HZ (outer)', 'Too cold'],
    default='No data'
)

# 6) Summary.
n_computed = in_range.sum()
print(f"   -> HZ limits computed for {n_computed} of {len(df)} planets.")
print("\n   hz_zone breakdown:")
print(df['hz_zone'].value_counts().to_string())

# Quick-look table: planets in the conservative HZ.
hz_planets = df[df['hz_zone'] == 'Conservative HZ'][[
    'name', 'star_name', 'star_teff', 'star_radius', 'hz_lum_solar',
    'hz_inner_conservative_au', 'hz_outer_conservative_au',
    'semi_major_axis',
]].copy()

print(f"\n   Planets in conservative HZ: {len(hz_planets)}")
hz_planets

6. Computing Habitable Zone limits (Kopparapu et al. 2014)...
   [!] 2 planets have T_eff outside 2600-7200 K. HZ columns will be NaN for those rows.
   -> HZ limits computed for 123 of 130 planets.

   hz_zone breakdown:
hz_zone
Too hot                  67
Too cold                 28
Conservative HZ          16
No data                  10
Optimistic HZ (inner)     7
Optimistic HZ (outer)     2

   Planets in conservative HZ: 16


,name,star_name,star_teff,star_radius,hz_lum_solar,hz_inner_conservative_au,hz_outer_conservative_au,semi_major_axis
5,82 Eri d,82 Eri,5368.0,0.9300,0.644326,0.781374,1.395058,1.35410
20,GJ 1002 b,GJ 1002,3024.0,0.1370,0.001408,0.039072,0.077640,0.04570
21,GJ 1002 c,GJ 1002,3024.0,0.1370,0.001408,0.039072,0.077640,0.07380
24,GJ 1061 d,GJ 1061,2953.0,0.1560,0.001660,0.042457,0.084640,0.05400
39,GJ 357 d,GJ 357,3505.0,0.3370,0.015378,0.128463,0.249458,0.20400
61,GJ 687 b,GJ 687,3389.0,0.4329,0.022179,0.154489,0.301711,0.16300
63,Wolf 1055 b,GJ 752 A,3534.0,0.4810,0.032378,0.186334,0.361316,0.34300
65,GJ 785 c,GJ 785,5181.0,0.8580,0.475903,0.678283,1.219011,1.18300
70,GJ 876 c,GJ 876,3293.7,0.3198,0.010799,0.107909,0.211719,0.13400
80,Gl 725 B c,Gl 725 B,3379.0,0.2800,0.009170,0.099346,0.194113,0.13900


## Step 7 — Theoretical Radius Fallback and Planet Size Category

This step is intentionally separated from the HZ computation.
Its purpose is structural characterization: define a robust planet mass/radius pair in Earth units, estimate a theoretical radius when no measured radius is available, and assign a size-based `planet_category` label.

Naming convention in this step:
- `*_th` means the value is part of the theoretical/fallback characterization workflow.
- Measured quantities are always preferred when available; inferred values are used only as fallback.

In [21]:
print("7. Building theoretical radius fallback and planet categories...")


def estimate_radius_from_mass(mass_earth):
    """
    Estimate planetary radius (Earth radii) from mass (Earth masses)
    using the piecewise relation adapted from Jared's mxlib reference.
    """
    if pd.isna(mass_earth) or mass_earth <= 0:
        return np.nan

    M = float(mass_earth)

    if M < 4.1:
        return M ** (1 / 3)

    if M < 15.84:
        return 0.62 * (M ** 0.67)

    if M < 3591.1:
        coeff = [14.0211, -44.8414, 53.6554, -25.3289, 5.4920, -0.4586]
        logM = np.log10(M)
        return sum(c * (logM ** p) for p, c in enumerate(coeff))

    return 32.03 * (M ** (-1 / 8))


# 1) Define the mass used by the theoretical/fallback workflow.
# Prefer true mass when available; otherwise use m*sin(i).
df['mass_earth_th'] = df['mass_earth'].where(df['mass_earth'].notna(), df['mass_sini_earth'])
df['mass_earth_th_source'] = np.select(
    [df['mass_earth'].notna(), df['mass_sini_earth'].notna()],
    ['mass_earth', 'mass_sini_earth'],
    default='missing'
)

# 2) Define the radius used by the theoretical/fallback workflow.
# Prefer measured radius; if missing, estimate from the fallback mass.
df['radius_earth_th'] = df['radius_earth']
radius_fill_mask = df['radius_earth_th'].isna() & df['mass_earth_th'].notna()
df.loc[radius_fill_mask, 'radius_earth_th'] = df.loc[radius_fill_mask, 'mass_earth_th'].apply(estimate_radius_from_mass)
df['radius_earth_th_source'] = np.select(
    [df['radius_earth'].notna(), radius_fill_mask],
    ['radius_earth', 'estimated_from_mass'],
    default='missing'
)

# 3) Planet size category from radius in Earth units.
r_th = pd.to_numeric(df['radius_earth_th'], errors='coerce')
df['planet_category'] = np.select(
    [
        r_th.notna() & (r_th < 1.6),
        r_th.notna() & (r_th < 4.0),
        r_th.notna() & (r_th < 8.0),
        r_th.notna(),
    ],
    ['Terrestrial', 'Sub-Neptune', 'Ice Giant', 'Gas Giant'],
    default='Unknown'
)

# 4) Summary.
print("   -> planet_category breakdown:")
print(df['planet_category'].value_counts().to_string())

th_preview = df[[
    'name',
    'mass_earth', 'mass_sini_earth', 'mass_earth_th', 'mass_earth_th_source',
    'radius_earth', 'radius_earth_th', 'radius_earth_th_source',
    'planet_category',
]].head(15).copy()

th_preview

7. Building theoretical radius fallback and planet categories...
   -> planet_category breakdown:
planet_category
Terrestrial    61
Sub-Neptune    48
Gas Giant      13
Ice Giant       8


C:\Users\Rafa\AppData\Local\Temp\ipykernel_30056\3911937739.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['radius_earth_th'] = df['radius_earth']
C:\Users\Rafa\AppData\Local\Temp\ipykernel_30056\3911937739.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['radius_earth_th_source'] = np.select(
C:\Users\Rafa\AppData\Local\Temp\ipykernel_30056\3911937739.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

,name,mass_earth,mass_sini_earth,mass_earth_th,mass_earth_th_source,radius_earth,radius_earth_th,radius_earth_th_source,planet_category
0,61 Vir b,NaN,5.784477,5.784477,mass_sini_earth,NaN,2.009595,estimated_from_mass,Sub-Neptune
1,61 Vir c,NaN,17.839708,17.839708,mass_sini_earth,NaN,4.355004,estimated_from_mass,Ice Giant
2,61 Vir d,NaN,10.920584,10.920584,mass_sini_earth,NaN,3.076213,estimated_from_mass,Sub-Neptune
3,82 Eri b,NaN,2.148520,2.148520,mass_sini_earth,NaN,1.290367,estimated_from_mass,Terrestrial
4,82 Eri c,NaN,2.981230,2.981230,mass_sini_earth,NaN,1.439235,estimated_from_mass,Terrestrial
5,82 Eri d,NaN,5.816260,5.816260,mass_sini_earth,NaN,2.016987,estimated_from_mass,Sub-Neptune
6,82 Eri e,NaN,4.767426,4.767426,mass_sini_earth,NaN,1.765395,estimated_from_mass,Sub-Neptune
7,AU Mic b,8.991366,NaN,8.991366,mass_earth,4.786235,4.786235,radius_earth,Ice Giant
8,AU Mic c,14.461193,NaN,14.461193,mass_earth,2.791036,2.791036,radius_earth,Sub-Neptune
9,AU Mic d,1.052012,NaN,1.052012,mass_earth,NaN,1.017045,estimated_from_mass,Terrestrial


## Output Verification

In [22]:
# Final export after all derived columns are ready (including planet_category).
df.to_csv(file_output, index=False)

print(f"Catalogue shape: {df.shape[0]} planets x {df.shape[1]} columns")
print(f"Output written to: {file_output}\n")

print("Detection method breakdown:")
print(df['Detection_Method_Full'].value_counts().to_string())

print("\nPlanet status breakdown:")
print(df['planet_status'].value_counts().to_string())

print("\nHZ zone breakdown:")
print(df['hz_zone'].value_counts().to_string())

print("\nPlanet category breakdown:")
print(df['planet_category'].value_counts().to_string())

print("\nFinal HZ + theoretical-size preview:")
print(df[[
    'name',
    'hz_inner_optimistic_au', 'hz_inner_conservative_au',
    'hz_outer_conservative_au', 'hz_outer_optimistic_au',
    'hz_zone',
    'Teq_val', 'Teq_error_min', 'Teq_error_max',
    'mass_earth_th', 'mass_earth_th_source',
    'radius_earth_th', 'radius_earth_th_source',
    'planet_category',
]].head(12).to_string(index=False))

Catalogue shape: 130 planets x 153 columns
Output written to: Exoplanets_SolarNeighbourhood_Catalogue.csv

Detection method breakdown:
Detection_Method_Full
RV                 112
Transit + RV        12
RV + Astrometry      4
TTV                  1
Astrometry           1

Planet status breakdown:
planet_status
Confirmed    101
Candidate     22
Discarded      7

HZ zone breakdown:
hz_zone
Too hot                  67
Too cold                 28
Conservative HZ          16
No data                  10
Optimistic HZ (inner)     7
Optimistic HZ (outer)     2

Planet category breakdown:
planet_category
Terrestrial    61
Sub-Neptune    48
Gas Giant      13
Ice Giant       8

Final HZ + theoretical-size preview:
            name  hz_inner_optimistic_au  hz_inner_conservative_au  hz_outer_conservative_au  hz_outer_optimistic_au         hz_zone     Teq_val  Teq_error_min  Teq_error_max  mass_earth_th mass_earth_th_source  radius_earth_th radius_earth_th_source planet_category
        61 Vir b    